# Median Filter

**Dataset**: PhysioNet Auditory EEG (Abo Alzahab et al., 2021)  
**Channels**: P4, Cz, F8, T7  
**Sampling rate**: 200 Hz  
**Subject**: 1

---

## Overview

The median filter replaces each sample with the median of its neighbors. It is resistant to outliers (spikes) because it is not affected by extreme values. We inject an artificial spike at sample 1000 to demonstrate this resistance.

## What you should expect to see

- The artificial spike appears clearly in the original signal
- The median filter removes the spike completely even with small windows
- Larger windows smooth the signal more

## Key parameters

| Parameter | Value | Meaning |
| --- | --- | --- |
| Channel | P4 | Parietal region |
| Sampling rate | 200 Hz | One sample every 5 ms |
| Windows | 5, 11, 21 | Window sizes |
| Spike location | 1000 | Sample 1000 |
| Spike value | +200 uV | Artificial addition |


## 1. Install dependencies


In [ ]:
!pip install scipy numpy plotly wfdb


## 2. Clone the resources repo and download one subject

We download only one subject (`--subjects 1`) to speed up the experiment in Colab.


In [ ]:
import os
if not os.path.exists('python-EEG-Arabic-Resources'):
    !git clone https://github.com/NibrasAz7/python-EEG-Arabic-Resources.git
os.chdir('python-EEG-Arabic-Resources')


In [ ]:
from pathlib import Path
data_dir = Path('data/local')
if not data_dir.exists() or not any(data_dir.glob('*.dat')):
    !python data/download_local.py --output data/local --subjects 1


## 3. Load the EEG signal

We load subject 1, experiment 1, session 2, channel **P4** (parietal region).


In [ ]:
import numpy as np
from utils.eeg_loader import load_local_eeg

timestamps, eeg_data, ch_names = load_local_eeg(
    data_dir='data/local', subject=1, experiment=1, session=2
)
channel_data = eeg_data[:, 0]  # P4 channel
fs = 200  # Sampling rate (Hz)

print(f'Channels: {ch_names}')
print(f'Signal length: {len(channel_data)} samples ({len(channel_data)/fs:.1f} seconds)')


## 4. Apply the filter

We inject an artificial spike (+200 uV) at sample 1000, then use `scipy.signal.medfilt` with three window sizes. The median effectively ignores extreme values.


In [ ]:
from scipy.signal import medfilt

spike_idx = 1000
channel_data = channel_data.copy()
channel_data[spike_idx] += 200.0

windows = [5, 11, 21]
filtered = {}
for w in windows:
    filtered[w] = medfilt(channel_data, kernel_size=w)
print(f'Injected spike at sample {spike_idx} (+200 uV)')
print(f'Applied median filter with windows: {windows}')


## 5. Interactive plot

**What to look for:**

- The red spike appears in the original signal (top)
- The median filter removes the spike completely in all windows
- Larger windows smooth the signal more but may hide real features
- Compare this with the other filters which spread the spike instead of removing it


In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

n_plot = min(5000, len(channel_data))
t_sec = timestamps[:n_plot] / 1000.0

fig = make_subplots(rows=4, cols=1, shared_xaxes=True,
                    subplot_titles=('Original with spike (P4)',
                                    'Median (window=5)',
                                    'Median (window=11)',
                                    'Median (window=21)'))
fig.add_trace(go.Scatter(x=t_sec, y=channel_data[:n_plot], name='Raw',
                         line=dict(color='gray', width=0.5)), row=1, col=1)
fig.add_trace(go.Scatter(x=[t_sec[spike_idx]], y=[channel_data[spike_idx]],
                         mode='markers', marker=dict(color='red', size=6),
                         name='Spike'), row=1, col=1)
for i, w in enumerate(windows, start=2):
    fig.add_trace(go.Scatter(x=t_sec, y=filtered[w][:n_plot],
                             name=f'w={w}', line=dict(width=0.5)), row=i, col=1)
fig.update_layout(height=900,
                  title_text='Median Filter - Channel P4 with Artificial Spike',
                  xaxis4_title='Time (s)', showlegend=False)
fig.show()


## What did we learn?

- The median filter replaces each sample with the median of its neighbors
- It resists outliers and spikes effectively
- It does not spread the spike like the moving average and Gaussian filters
- It is well suited for removing spikes before applying other filters
